# Unified Classification & Regression Pipeline on SPY

In [3]:
import pandas as pd
import numpy as np
import yfinance as yf
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, mean_absolute_error, r2_score
import joblib

In [4]:
# 1. Download SPY data (6 months)
df = yf.download("SPY", period="6mo", interval="1d")
df.dropna(inplace=True)


YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed


In [5]:
# Feature engineering (candlestick + momentum + RSI)
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(period).mean()
    loss = -delta.clip(upper=0).rolling(period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def generate_candle(df):
    df = df.copy()
    df['Body'] = (df['Close'] - df['Open']).abs()
    df['UT'] = df['High'] - np.maximum(df['Open'], df['Close'])
    df['LT'] = np.minimum(df['Open'], df['Close']) - df['Low']
    df['Range'] = df['High'] - df['Low']
    for c in ['Body','UT','LT']:
        df[c + '_pct'] = df[c] / df['Range']
        df[c + '_MA'] = df[c].rolling(20).mean()
    df['AllowTrade'] = (
        (df['Body'] > df['Body_MA']) |
        (df['UT'] > df['UT_MA']) |
        (df['LT'] > df['LT_MA'])
    ).astype(int)
    return df

def add_momentum(df):
    df = df.copy()
    df['MA5'] = df['Close'].rolling(5).mean()
    df['MA20'] = df['Close'].rolling(20).mean()
    df['Momentum_5'] = df['Close'] - df['Close'].shift(5)
    df['RSI_14'] = compute_rsi(df['Close'], 14)
    return df

df = generate_candle(df)
df = add_momentum(df)
df.dropna(inplace=True)
features = ['Body_pct','UT_pct','LT_pct','Momentum_5','MA5','MA20','RSI_14']


In [17]:
# -- Classification: Predict Direction (Fixed for MultiIndex) --

# If df has MultiIndex columns, flatten them
if isinstance(df.columns, pd.MultiIndex):
    df.columns = ['_'.join(col).strip() for col in df.columns.values]

# Example: now 'Close_SPY' instead of ('Close', 'SPY')
# Adjust all feature and label references accordingly

df_cls = df.copy()

# Use the correct flattened Close column
close_col = [col for col in df_cls.columns if 'Close' in col][0]
df_cls['Future_Close'] = df_cls[close_col].shift(-1)
df_cls['Price_Change'] = df_cls['Future_Close'] - df_cls[close_col]

# Recreate features list using the suffix (e.g., _SPY)
features = [col for col in df_cls.columns if any(f in col for f in [
    'Body_pct', 'UT_pct', 'LT_pct', 'Momentum_5', 'MA5', 'MA20', 'RSI_14'
])]


# Label direction based on price movement
df_cls['direction'] = np.where(df_cls['Price_Change'] > 0.5, 'Up',
                        np.where(df_cls['Price_Change'] < -0.5, 'Down', 'Flat'))

# Drop last row with no future price
df_cls.dropna(subset=['direction'], inplace=True)

# Use the same features as defined earlier
Xc = df_cls[features]
yc = df_cls['direction']

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import joblib

le_cls = LabelEncoder()
yc_enc = le_cls.fit_transform(yc)

Xtc, Xtc_test, ytc, ytc_test = train_test_split(Xc, yc_enc, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42)
clf.fit(Xtc, ytc)

# Evaluate
y_predc = clf.predict(Xtc_test)
print("Classification Report for Direction:")
print(classification_report(ytc_test, y_predc, target_names=le_cls.classes_))

# Save
joblib.dump(clf, "direction_classifier.pkl")
joblib.dump(le_cls, "direction_label_encoder.pkl")



Classification Report for Direction:
              precision    recall  f1-score   support

        Down       0.50      0.29      0.36         7
        Flat       0.00      0.00      0.00         3
          Up       0.59      0.91      0.71        11

    accuracy                           0.57        21
   macro avg       0.36      0.40      0.36        21
weighted avg       0.47      0.57      0.50        21



/home/hiren/local/arh_venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/hiren/local/arh_venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/hiren/local/arh_venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


['direction_label_encoder.pkl']

In [18]:
# -- Regression: Predict Price Change --
df_reg = df.copy()
df_reg['Target_PriceChange'] = df_reg['Close'].shift(-1) - df_reg['Close']
df_reg.dropna(subset=['Target_PriceChange'], inplace=True)

Xr = df_reg[features]
yr = df_reg['Target_PriceChange']
Xtr, Xtr_test, ytr, ytr_test = train_test_split(Xr, yr, test_size=0.2, random_state=42)
reg = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
reg.fit(Xtr, ytr)
y_predr = reg.predict(Xtr_test)
print("Regression Results:")
print(f"MAE: {mean_absolute_error(ytr_test, y_predr):.4f}")
print(f"R²: {r2_score(ytr_test, y_predr):.4f}")
joblib.dump(reg, "price_regressor.pkl")

KeyError: 'Close'